# Dataset Generation for Cipher Classification
Dataset Generation for Cipher Classification

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import pandas as pd
import random
import string

from src.ciphers import caesar, affine, vigenere, substitution, columnar_transposition, playfair
from src.features.extractor import FeatureExtractor

## Load English Corpus

In [ ]:
text_corpus = """It is a truth universally acknowledged, that a single man in possession of a good fortune, must be in want of a wife.
However little known the feelings or views of such a man may be on his first entering a neighbourhood, this truth is so well fixed in the minds of the surrounding families, that he is considered the rightful property of some one or other of their daughters.
"My dear Mr. Bennet," said his lady to him one day, "have you heard that Netherfield Park is let at last?"
Mr. Bennet replied that he had not.
"But it is," returned she; "for Mrs. Long has just been here, and she told me all about it."
Mr. Bennet made no answer.
"Do you not want to know who has taken it?" cried his wife impatiently.
"You want to tell me, and I have no objection to hearing it."
This was invitation enough.
"Why, my dear, you must know, Mrs. Long says that Netherfield is taken by a young man of large fortune from the north of England; that he came down on Monday in a chaise and four to see the place, and was so much delighted with it, that he agreed with Mr. Morris immediately; that he is to take possession before Michaelmas, and some of his servants are to be in the house by the end of next week."
"What is his name?"
"Bingley."
"Is he married or single?"
"Oh! Single, my dear, to be sure! A single man of large fortune; four or five thousand a year. What a fine thing for our girls!"
"How so? How can it affect them?"
"My dear Mr. Bennet," replied his wife, "how can you be so tiresome! You must know that I am thinking of his marrying one of them."
"Is that his design in settling here?"
"Design! Nonsense, how can you talk so! But it is very likely that he may fall in love with one of them, and therefore you must visit him as soon as he comes."
"I see no occasion for that. You and the girls may go, or you may send them by themselves, which perhaps will be still better, for as you are as handsome as any of them, Mr. Bingley may like you the best of the party."
"My dear, you flatter me. I certainly have had my share of beauty, but I do not pretend to be anything extraordinary now. When a woman has five grown-up daughters, she ought to give over thinking of her own beauty."
"In such cases, a woman has not often much beauty to think of."
"But, my dear, you must indeed go and see Mr. Bingley when he comes into the neighbourhood."
"It is more than I engage for, I assure you."
"But consider your daughters. Only think what an establishment it would be for one of them. Sir William and Lady Lucas are determined to go, merely on that account, for in general, you know, they visit no newcomers. Indeed you must go, for it will be impossible for us to visit him if you do not."
"You are over-scrupulous, surely. I dare say Mr. Bingley will be very glad to see you; and I will send a few lines by you to assure him of my hearty consent to his marrying whichever he chooses of the girls; though I must throw in a good word for my little Lizzy."
"I desire you will do no such thing. Lizzy is not a bit better than the others; and I am sure she is not half so handsome as Jane, nor half so good-humoured as Lydia. But you are always giving her the preference."
"They have none of them much to recommend them," replied he; "they are all silly and ignorant like other girls; but Lizzy has something more of quickness than her sisters."
"Mr. Bennet, how can you abuse your own children in such a way? You take delight in vexing me. You have no compassion for my poor nerves."
"You mistake me, my dear. I have a high respect for your nerves. They are my old friends. I have heard you mention them with consideration these last twenty years at least."
"""

## Text Preprocessing
Clean text to uppercase A-Z only, split into chunks of varying sizes (100-500 chars)

In [ ]:
import re
def clean_text(text):
    return re.sub(r'[^A-Z]', '', text.upper())

cleaned_corpus = clean_text(text_corpus)
chunks = []
idx = 0
while idx < len(cleaned_corpus) - 100:
    chunk_size = random.randint(100, 500)
    chunk = cleaned_corpus[idx:idx+chunk_size]
    if len(chunk) >= 100:
        chunks.append(chunk)
    idx += chunk_size
print(f'Generated {len(chunks)} chunks.')

## Generate Cipher Samples
For each cipher type, encrypt chunks with random keys:
- Caesar: random shift 1-25
- Affine: random valid (a,b) pairs
- Vigenère: random keywords length 3-7
- Substitution: random permutation keys
- Columnar Transposition: random keywords length 3-7
- Playfair: random keywords
Generate 500 samples per cipher type = 3000 total

In [ ]:
samples = []
cipher_types = ['caesar', 'affine', 'vigenere', 'substitution', 'columnar_transposition', 'playfair']
num_samples_per_cipher = 500

def get_random_chunk():
    if chunks:
        return random.choice(chunks)
    return 'A'*100

def generate_random_substitution_key():
    alpha = list(string.ascii_uppercase)
    random.shuffle(alpha)
    return ''.join(alpha)

for _ in range(num_samples_per_cipher):
    # Caesar
    pt = get_random_chunk()
    shift = random.randint(1, 25)
    ct_caesar = caesar.encrypt(pt, shift)
    samples.append({'plaintext': pt, 'ciphertext': ct_caesar, 'label': 'caesar'})
    
    # Affine
    pt = get_random_chunk()
    valid_a = [1, 3, 5, 7, 9, 11, 15, 17, 19, 21, 23, 25]
    a = random.choice(valid_a)
    b = random.randint(0, 25)
    ct_affine = affine.encrypt(pt, a, b)
    samples.append({'plaintext': pt, 'ciphertext': ct_affine, 'label': 'affine'})
    
    # Vigenere
    pt = get_random_chunk()
    kw_len = random.randint(3, 7)
    kw = ''.join(random.choices(string.ascii_uppercase, k=kw_len))
    ct_vig = vigenere.encrypt(pt, kw)
    samples.append({'plaintext': pt, 'ciphertext': ct_vig, 'label': 'vigenere'})
    
    # Substitution
    pt = get_random_chunk()
    key_sub = generate_random_substitution_key()
    ct_sub = substitution.encrypt(pt, key_sub)
    samples.append({'plaintext': pt, 'ciphertext': ct_sub, 'label': 'substitution'})
    
    # Columnar Transposition
    pt = get_random_chunk()
    kw_len = random.randint(3, 7)
    kw = ''.join(random.choices(string.ascii_uppercase, k=kw_len))
    ct_col = columnar_transposition.encrypt(pt, kw)
    samples.append({'plaintext': pt, 'ciphertext': ct_col, 'label': 'columnar_transposition'})
    
    # Playfair
    pt = get_random_chunk()
    kw_len = random.randint(3, 10)
    kw = ''.join(random.choices(string.ascii_uppercase, k=kw_len))
    pt_playfair = pt.replace('J', 'I')
    if len(pt_playfair) % 2 != 0: pt_playfair += 'X'
    ct_play = playfair.encrypt(pt_playfair, kw)
    samples.append({'plaintext': pt_playfair, 'ciphertext': ct_play, 'label': 'playfair'})

print(f'Generated {len(samples)} total samples.')

## Feature Extraction
Extract features for all samples using FeatureExtractor

In [ ]:
extractor = FeatureExtractor()
dataset_rows = []

for s in samples:
    ct = s['ciphertext']
    features = extractor.extract_all_features(ct)
    features['label'] = s['label']
    dataset_rows.append(features)
    
df = pd.DataFrame(dataset_rows)
df.head()

## Save Dataset
Save to CSV with features and labels

In [ ]:
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/cipher_dataset.csv', index=False)
print('Dataset saved to ../data/processed/cipher_dataset.csv')

## Dataset Summary
Print counts, shapes, label distribution

In [ ]:
print(f'Dataset shape: {df.shape}')
print('\nLabel distribution:')
print(df['label'].value_counts())